In [5]:
from rdflib import Graph, URIRef, Literal, RDFS
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt

# ---------- 1) Load RDF/Turtle ----------
ttl_path = "./datasets/pmdco2_tto_example_diagonal.ttl"   # <- point to your file
g = Graph()
g.parse(ttl_path, format="turtle")

# ---------- 2) Helpers ----------
def qlabel(node: object, g: Graph):
    """Human-ish label for URIs/literals."""
    if isinstance(node, Literal):
        return str(node)[:80]
    if isinstance(node, URIRef):
        # rdfs:label > qname > short URI tail
        for _, _, lab in g.triples((node, RDFS.label, None)):
            return str(lab)
        try:
            return g.namespace_manager.normalizeUri(node)
        except Exception:
            s = str(node)
            return s.rsplit("/", 1)[-1].rsplit("#", 1)[-1]
    return str(node)

def to_networkx(g: Graph, include_literals=False, predicate_whitelist=None):
    """
    Convert RDF graph to a MultiDiGraph.
    - include_literals: add Literal objects as nodes (often noisy).
    - predicate_whitelist: set of predicates to keep (URIRefs); None keeps all.
    """
    G = nx.MultiDiGraph()
    for s, p, o in g:
        if predicate_whitelist and p not in predicate_whitelist:
            continue
        s_lbl = qlabel(s, g)
        p_lbl = qlabel(p, g)
        if isinstance(o, Literal) and not include_literals:
            # attach literal as edge attribute (label/value) instead of a node
            G.add_node(s_lbl)
            G.nodes[s_lbl].setdefault("data", {})[p_lbl] = str(o)
        else:
            o_lbl = qlabel(o, g)
            G.add_node(s_lbl)
            G.add_node(o_lbl)
            G.add_edge(s_lbl, o_lbl, label=p_lbl)
    return G

# ---------- 3a) Quick static drawing (good for small graphs) ----------
def draw_static(G: nx.Graph):
    pos = nx.spring_layout(G, k=0.8, seed=42)  # simple force layout
    edge_labels = {(u, v): d.get("label", "") for u, v, d in G.edges(data=True)}
    plt.figure(figsize=(10, 8))
    nx.draw(G, pos, with_labels=True, node_size=600, font_size=8)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    plt.tight_layout()
    plt.show()

# ---------- 3b) Interactive HTML (best for exploring) ----------
def draw_interactive(G: nx.Graph, html_out="knowledgegraph.html"):
    net = Network(height="750px", width="100%", directed=True, notebook=False)
    net.toggle_physics(True)
    # move node attributes into the UI
    for n, attrs in G.nodes(data=True):
        title = "<br>".join(f"{k}: {v}" for k, v in attrs.get("data", {}).items()) if "data" in attrs else ""
        net.add_node(n, label=n, title=title)
    for u, v, d in G.edges(data=True):
        net.add_edge(u, v, label=d.get("label", ""))
    net.show(html_out)
    print(f"Saved interactive visualization to: {html_out}")

# ---------- 4) Build + visualize ----------
G = to_networkx(g, include_literals=False)     # start tidy: literals as attributes
# draw_static(G)                               # uncomment for a quick PNG-ish view
draw_interactive(G, "knowledgegraph.html")     # open the HTML in your browser

knowledgegraph.html


AttributeError: 'NoneType' object has no attribute 'render'